In [1]:
import pandas as pd
import os

df_ratings = pd.read_csv('ml-latest-small/ratings.csv')

print(f'Total de avaliações: {len(df_ratings)}')
print(f'Total de usuários: {df_ratings["userId"].nunique()}')
print(f'Total de filmes: {df_ratings["movieId"].nunique()}')
df_ratings.head()

Total de avaliações: 100836
Total de usuários: 610
Total de filmes: 9724


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [2]:
def split_por_usuario(df, test_size=0.3, random_state=42):
    """
    Para cada usuário, sorteia aleatoriamente 30% das avaliações para teste
    e mantém os 70% restantes para treino.
    """
    train_list = []
    test_list = []

    for usuario_id, grupo in df.groupby('userId'):
        grupo_shuffled = grupo.sample(frac=1, random_state=random_state)
        n_test = max(1, round(len(grupo_shuffled) * test_size))

        test_list.append(grupo_shuffled.iloc[:n_test])
        train_list.append(grupo_shuffled.iloc[n_test:])

    df_train = pd.concat(train_list).reset_index(drop=True)
    df_test = pd.concat(test_list).reset_index(drop=True)

    return df_train, df_test


df_train, df_test = split_por_usuario(df_ratings, test_size=0.3, random_state=42)

print(f'Treino: {len(df_train)} avaliações ({len(df_train)/len(df_ratings)*100:.1f}%)')
print(f'Teste:  {len(df_test)} avaliações ({len(df_test)/len(df_ratings)*100:.1f}%)')

Treino: 70595 avaliações (70.0%)
Teste:  30241 avaliações (30.0%)


In [3]:
# Verifica se o split por usuário está correto
verificacao = df_ratings.groupby('userId').size().rename('total')
verificacao = verificacao.to_frame()
verificacao['treino'] = df_train.groupby('userId').size()
verificacao['teste'] = df_test.groupby('userId').size()
verificacao['pct_teste'] = (verificacao['teste'] / verificacao['total'] * 100).round(1)

print(f'% média de teste por usuário: {verificacao["pct_teste"].mean():.1f}%')
print(f'Min: {verificacao["pct_teste"].min()}%  |  Max: {verificacao["pct_teste"].max()}%')
verificacao.head(10)

% média de teste por usuário: 30.0%
Min: 28.6%  |  Max: 32.0%


,total,treino,teste,pct_teste
userId,,,,
1,232,162,70,30.2
2,29,20,9,31.0
3,39,27,12,30.8
4,216,151,65,30.1
5,44,31,13,29.5
6,314,220,94,29.9
7,152,106,46,30.3
8,47,33,14,29.8
9,46,32,14,30.4


In [4]:
os.makedirs('data_spliting_cf1', exist_ok=True)

df_train.to_csv('data_spliting_cf1/training.csv', index=False)
df_test.to_csv('data_spliting_cf1/testing.csv', index=False)

print('Arquivos salvos em data_spliting_cf1/')
print(f'  training.csv: {len(df_train)} linhas')
print(f'  testing.csv:  {len(df_test)} linhas')

Arquivos salvos em data_spliting_cf1/
  training.csv: 70595 linhas
  testing.csv:  30241 linhas
